# Foundation-Model-Assisted Niche Consistency Across Independent Cohort Halves

## Abstract
Spatial transcriptomics provides biological supervision but is difficult to scale across large pathology cohorts. This notebook evaluates whether niche structure derived from H&E remains consistent when transferring from the first half of a cohort to the second half, and whether foundation-model morphology features improve that consistency. We compare a baseline pipeline (composition-only) against a foundation-model-assisted pipeline (composition + Hoptimus-0). GOEA/LLM labels from reference-crosscohort-label-transfer are used as fixed reference outcomes and are not re-derived here.

## Introduction
### Motivation
A core translational question is not only whether niches can be discovered, but whether their biological signal is stable across independent portions of the cohort. If niche definitions drift strongly between halves, downstream interpretation and deployment become fragile.

### Study Objective
This notebook quantifies cross-half niche consistency under matched settings and reports the consistency delta attributable to foundation-model morphology features.

### Scope (Important)
- We use reference labels from GOEA/LLM outputs (from reference-crosscohort-label-transfer) as fixed outcomes.
- We do not re-derive or re-implement GOEA/LLM methodology here.
- We compare two niche pipelines:
  - Baseline: composition-only niche features
  - FM: composition + foundation-model morphology (Hoptimus)

Primary question: does FM-assisted niche discovery improve cross-half consistency?

## Cohort and Split
- 4 samples x (Top, Mid, Bot) = 12 portions (P1-P12).
- First half: P1-P6 (discovery side).
- Second half: P7-P12 (validation side).

## Outputs
1. Niche alignment between first half and second half.
2. Marker consistency and distribution consistency per matched niche.
3. Baseline vs FM delta summary tables and figures.

## Stage 0 · Study Design

### 0-A: Research Question
Can niche patterns discovered from H&E remain consistent when moving from the first half of the cohort (P1-P6) to the second half (P7-P12), and does foundation-model assistance improve this consistency?

### 0-B: Why This Notebook Is Different from reference-crosscohort-label-transfer
- reference-crosscohort-label-transfer established GOEA+LLM-based niche naming.
- This notebook does not duplicate that innovation.
- Here, GOEA/LLM outcomes are treated as fixed reference labels, while the main objective is consistency quantification.

### 0-C: Experimental Arms
1. Baseline: composition-only niche features.
2. FM-assisted: composition + Hoptimus morphology features.

Both arms use matched settings so that metric deltas are interpretable.

## Stage 0 · Notebook Roadmap

This notebook is organized as:
1. Environment and configuration.
2. Optional niche recomputation (baseline vs FM).
3. Cohort harmonization and niche alignment.
4. Consistency scoring and visualization.
5. Optional classification-style reporting (AUC/CM) when probabilistic predictions are available.

Each code block below is preceded by a short methods note to make the computation-to-interpretation link explicit.

In [ ]:
# Optional installs (safe to rerun). Uncomment if needed.
# %pip install -q pandas numpy scipy scikit-learn matplotlib seaborn

### 0-E: Dependencies

This cell is a minimal package checkpoint. It keeps environment setup explicit and reproducible before any scientific computation.

In [ ]:
from __future__ import annotations

import os
import json
import subprocess
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd

from scipy.stats import ttest_ind
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_context('talk')
sns.set_style('whitegrid')

print('Imports OK')

### 0-F: Core Imports

This cell imports numerical, statistical, optimization, and plotting libraries used throughout the notebook:
- statistics and hypothesis testing,
- distance-based matching,
- and publication-style visualization.

In [ ]:
# ---------------------------
# Configuration (auto-detect first, then fallback)
# ---------------------------
PROJECT_ROOT = Path('/workspace/wsinsight/devel/wsinsight')


def first_existing(candidates: list[Path], want_dir: bool | None = None) -> Path | None:
    for p in candidates:
        if p.exists():
            if want_dir is None:
                return p
            if want_dir and p.is_dir():
                return p
            if (not want_dir) and p.is_file():
                return p
    return None


# Candidate slide sources. Prefer reference-crosscohort-label-transfer style data paths when available.
WSI_CANDIDATES = [
    PROJECT_ROOT / 'data',
    PROJECT_ROOT / 'slides',
    PROJECT_ROOT / '12',
]
WSI_SOURCE = first_existing(WSI_CANDIDATES, want_dir=True)
if WSI_SOURCE is None:
    WSI_SOURCE = PROJECT_ROOT / 'CHANGE_ME_wsi_dir'

# Existing results directory that already contains model-outputs-csv from wsinsight run/infer.
RESULTS_SOURCE_CANDIDATES = [
    PROJECT_ROOT / 'outputs' / 'v3',
    PROJECT_ROOT / 'outputs',
    PROJECT_ROOT,
]

SOURCE_RESULTS_DIR = None
for root in RESULTS_SOURCE_CANDIDATES:
    cand = root / 'model-outputs-csv'
    if cand.exists() and cand.is_dir():
        SOURCE_RESULTS_DIR = root
        break
if SOURCE_RESULTS_DIR is None:
    SOURCE_RESULTS_DIR = PROJECT_ROOT / 'CHANGE_ME_results_dir_with_model_outputs_csv'

# Separate result roots to avoid baseline/FM niche outputs overriding each other.
RESULTS_BASELINE = PROJECT_ROOT / 'results-consistency-baseline'
RESULTS_FM = PROJECT_ROOT / 'results-consistency-fm'

# Reference GOEA/LLM-derived label file from reference-crosscohort-label-transfer outputs.
# This notebook uses those labels as fixed references (no GOEA/LLM re-derivation here).
REF_LABEL_CANDIDATES = [
    PROJECT_ROOT / 'reference-crosscohort-label-transfer-labels.csv',
    PROJECT_ROOT / 'outputs' / 'v3' / 'v2_g1a_goea' / 'niche_labels.csv',
    PROJECT_ROOT / 'outputs' / 'v3' / 'v2_g1a_goea' / 'predicted_niche_labels.csv',
]
REF_LABELS_CSV = first_existing(REF_LABEL_CANDIDATES, want_dir=False)
if REF_LABELS_CSV is None:
    REF_LABELS_CSV = PROJECT_ROOT / 'CHANGE_ME_reference_v2_labels.csv'

# Portion and split definition.
PORTIONS = [f'P{i}' for i in range(1, 13)]
FIRST_HALF = set(PORTIONS[:6])
SECOND_HALF = set(PORTIONS[6:])

# Run commands from notebook?
RUN_COMMANDS = False

# Differential marker settings
TOP_MARKERS = 30
EPS = 1e-8

print('Config ready')
print('WSI_SOURCE:', WSI_SOURCE)
print('SOURCE_RESULTS_DIR:', SOURCE_RESULTS_DIR)
print('RESULTS_BASELINE:', RESULTS_BASELINE)
print('RESULTS_FM:', RESULTS_FM)
print('REF_LABELS_CSV:', REF_LABELS_CSV)
print('Tips: If any path shows CHANGE_ME, edit this cell once before running downstream cells.')

### 0-G: Configuration and Path Resolution

This cell centralizes all paths and execution switches. It tries auto-detection first, then falls back to editable placeholders, so the same notebook can run across machines with minimal edits.

## Optional: Run WSInsight Niche Pipelines

This notebook can consume existing outputs or launch two runs:
- Baseline (composition-only)
- FM (composition + Hoptimus)

Note: this section does not implement GOEA/LLM. It only runs niche discovery.

### 0-D: Practical Path Logic

This notebook supports two execution modes:
- Reuse existing WSInsight outputs (fast path).
- Launch niche runs from this notebook (recompute path).

To avoid file collisions, baseline and FM outputs are written to separate result folders. If a configured path is missing, the config cell prints a CHANGE_ME hint so you can correct it once and continue.

### 0-H: Method Bridge to WSInsight Internals (DGI, kNN, Leiden/KMeans, PCA, Hoptimus-0)

The `wsinsight niche` command in the next code cell encapsulates these components:

1. kNN graph construction from cell-neighborhood context.
2. DGI embedding on the graph to learn compact niche-relevant representations.
3. Clustering by Leiden sweep (auto-k) or KMeans (fixed-k) to assign niche IDs.
4. Optional Hoptimus-0 morphology features, reduced by PCA before concatenation.

Conceptually:
- kNN defines local structure.
- DGI learns representation $z_i$ for each cell node.
- Leiden/KMeans groups similar $z_i$ into niche communities.
- Hoptimus + PCA adds morphology signal while controlling dimensionality.

In [ ]:
def build_niche_cmd(results_dir: Path, use_hoptimus: bool) -> list[str]:
    cmd = [
        'wsinsight', 'niche',
        '--wsi-dir', str(WSI_SOURCE),
        '--results-dir', str(results_dir),
        '--k-hops', '2',
        '--max-edge-len-um', '25.0',
        '--embed-dim', '32',
        '--epochs', '300',
        '--patience', '20',
        '--min-delta', '1e-4',
        '--min-epochs', '50',
        '--seed', '0',
    ]
    if use_hoptimus:
        cmd += ['--hoptimus', '--hoptimus-pca-dim', '32']
    return cmd


def ensure_results_scaffold(target_results_dir: Path, source_results_dir: Path) -> None:
    """
    Prepare a results dir for niche command by making sure model-outputs-csv exists.
    Baseline and FM runs are separated to prevent output overwrite.
    """
    import shutil

    target_results_dir.mkdir(parents=True, exist_ok=True)

    src_model = source_results_dir / 'model-outputs-csv'
    dst_model = target_results_dir / 'model-outputs-csv'

    if dst_model.exists():
        return

    if not src_model.exists():
        raise FileNotFoundError(
            f'Missing source model outputs: {src_model}. '
            'Set SOURCE_RESULTS_DIR to a folder that contains model-outputs-csv.'
        )

    # Try symlink first (fast, saves space). Fallback to copytree if symlink fails.
    try:
        dst_model.symlink_to(src_model, target_is_directory=True)
        print(f'[scaffold] symlinked {dst_model} -> {src_model}')
    except Exception:
        shutil.copytree(src_model, dst_model)
        print(f'[scaffold] copied {src_model} -> {dst_model}')


def run_cmd(cmd: list[str]) -> None:
    print('\n$', ' '.join(cmd))
    subprocess.run(cmd, check=True)


if RUN_COMMANDS:
    ensure_results_scaffold(RESULTS_BASELINE, SOURCE_RESULTS_DIR)
    ensure_results_scaffold(RESULTS_FM, SOURCE_RESULTS_DIR)

    run_cmd(build_niche_cmd(RESULTS_BASELINE, use_hoptimus=False))
    run_cmd(build_niche_cmd(RESULTS_FM, use_hoptimus=True))
else:
    print('RUN_COMMANDS=False: expecting existing outputs in configured result folders.')
    print('If first-time run, set RUN_COMMANDS=True after confirming WSI_SOURCE and SOURCE_RESULTS_DIR.')

## Stage 1 · Data Intake and Cohort Harmonization

### 1-A: What Is Loaded
For each slide CSV under niche outputs:
- per-cell niche one-hot columns (`niche_*`)
- model cell-type probabilities (`prob_*`)
- optional expression channels (`expr_*`)

### 1-B: Split Assignment
Slides are mapped to P1-P12 via stem parsing, then grouped into:
- first6: P1-P6
- next6: P7-P12

### 1-C: Why This Step Matters
Consistency analysis is only meaningful if split assignment is deterministic and reproducible. This stage guarantees that all downstream metrics compare the same cohort partition.

In [ ]:
def stem_to_portion(stem: str) -> str:
    """Infer P1..P12 from known stem patterns; fallback to stem itself."""
    mapping = {
        'S1_Top': 'P1', 'S1_Mid': 'P2', 'S1_Bot': 'P3',
        'S2_Top': 'P4', 'S2_Mid': 'P5', 'S2_Bot': 'P6',
        'S3_Top': 'P7', 'S3_Mid': 'P8', 'S3_Bot': 'P9',
        'S4_Top': 'P10', 'S4_Mid': 'P11', 'S4_Bot': 'P12',
    }
    for k, v in mapping.items():
        if k.lower() in stem.lower():
            return v
    for p in PORTIONS:
        if p.lower() in stem.lower():
            return p
    return stem


def infer_half(stem: str, portion: str) -> str:
    if portion in FIRST_HALF:
        return 'first6'
    if portion in SECOND_HALF:
        return 'next6'

    s = stem.lower()
    if 's1_' in s or 's2_' in s:
        return 'first6'
    if 's3_' in s or 's4_' in s:
        return 'next6'
    return 'unknown'


def detect_niche_cols(df: pd.DataFrame) -> list[str]:
    return sorted([c for c in df.columns if c.startswith('niche_') and c[6:].isdigit()],
                  key=lambda x: int(x.split('_')[1]))


def detect_expr_cols(df: pd.DataFrame) -> list[str]:
    return sorted([c for c in df.columns if c.startswith('expr_')])


def detect_prob_cols(df: pd.DataFrame) -> list[str]:
    return sorted([c for c in df.columns if c.startswith('prob_')])


def read_cells_table(results_dir: Path) -> pd.DataFrame:
    cells_dir = results_dir / 'niche-outputs-csv' / 'cells'
    if not cells_dir.exists():
        raise FileNotFoundError(f'Missing cells dir: {cells_dir}')

    rows = []
    for fp in sorted(cells_dir.glob('*.csv')):
        d = pd.read_csv(fp)
        d['slide_stem'] = fp.stem
        d['portion'] = stem_to_portion(fp.stem)
        d['half'] = infer_half(fp.stem, d['portion'].iloc[0] if len(d) else 'unknown')
        rows.append(d)

    if not rows:
        raise ValueError(f'No CSV files found under {cells_dir}')

    out = pd.concat(rows, ignore_index=True)
    return out


def load_reference_labels(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        print(f'[WARN] Reference label file not found: {path}')
        return None
    df = pd.read_csv(path)

    niche_col_candidates = ['niche_id', 'niche_group', 'niche']
    label_col_candidates = ['predicted_label', 'label', 'niche_label']

    niche_col = next((c for c in niche_col_candidates if c in df.columns), None)
    label_col = next((c for c in label_col_candidates if c in df.columns), None)

    if niche_col is None or label_col is None:
        print('[WARN] Could not find niche+label columns in reference label file.')
        return None

    ref = df[[niche_col, label_col]].copy()
    ref.columns = ['niche_key', 'ref_label']
    ref['niche_key'] = ref['niche_key'].astype(str)
    return ref.drop_duplicates()

### 1-D: Parsing and Harmonization Helpers

This helper block standardizes slide identifiers, split assignment, and required feature column detection so downstream statistics operate on a consistent schema.

In [ ]:
baseline_df = read_cells_table(RESULTS_BASELINE)
fm_df = read_cells_table(RESULTS_FM)
ref_labels = load_reference_labels(REF_LABELS_CSV)

print('Baseline rows:', len(baseline_df))
print('FM rows:', len(fm_df))
print('Baseline halves:', baseline_df['half'].value_counts(dropna=False).to_dict())
print('FM halves:', fm_df['half'].value_counts(dropna=False).to_dict())
print('Reference labels loaded:', 0 if ref_labels is None else len(ref_labels))

### 2-A: Alignment Readout Checkpoint

This checkpoint confirms that harmonized tables and reference labels loaded successfully before entering centroid matching and consistency scoring.

## Stage 2 · Cross-Half Niche Alignment

Niche IDs are not guaranteed to be semantically aligned across first6 and next6. Therefore, we align niches by feature similarity before any consistency score is computed.

### 2-A: Centroid Construction
For each niche, build a centroid from available feature blocks (`prob_*` and, when available, `expr_*`).

### 2-B: Matching Rule
Let $C^{(1)}_i$ be centroid $i$ from first6 and $C^{(2)}_j$ be centroid $j$ from next6. We form a cosine-distance cost matrix:
$$
D_{ij} = 1 - \frac{\langle C^{(1)}_i, C^{(2)}_j \rangle}{\|C^{(1)}_i\|_2\,\|C^{(2)}_j\|_2}
$$
Then we apply Hungarian matching to obtain one-to-one niche pairing with minimal global cost.

### 2-C: Output of This Stage
A matched niche-pair table used as the backbone for all consistency metrics.

In [ ]:
def assign_niche_id(df: pd.DataFrame) -> pd.DataFrame:
    niche_cols = detect_niche_cols(df)
    if not niche_cols:
        raise ValueError('No niche_* columns found.')
    x = df[niche_cols].to_numpy(dtype=float)
    idx = np.argmax(x, axis=1)
    out = df.copy()
    out['niche_id'] = idx
    out['niche_name'] = ['niche_' + str(i) for i in idx]
    return out

def niche_centroids(df: pd.DataFrame) -> pd.DataFrame:
    prob_cols = detect_prob_cols(df)
    expr_cols = detect_expr_cols(df)

    feat_cols = prob_cols + expr_cols
    if not feat_cols:
        raise ValueError('Need prob_* and/or expr_* columns for centroid alignment.')

    g = df.groupby('niche_id', observed=True)[feat_cols].mean()
    g['n_cells'] = df.groupby('niche_id', observed=True).size()
    g = g.reset_index()
    return g

def align_niches(first_cent: pd.DataFrame, second_cent: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    a = first_cent[feature_cols].to_numpy()
    b = second_cent[feature_cols].to_numpy()

    # Cosine distance; lower is better.
    cost = cdist(a, b, metric='cosine')
    r, c = linear_sum_assignment(cost)

    rec = []
    for i, j in zip(r, c):
        rec.append({
            'niche_first6': int(first_cent.iloc[i]['niche_id']),
            'niche_next6': int(second_cent.iloc[j]['niche_id']),
            'cosine_distance': float(cost[i, j]),
            'cosine_similarity': float(1.0 - cost[i, j]),
            'n_cells_first6': int(first_cent.iloc[i]['n_cells']),
            'n_cells_next6': int(second_cent.iloc[j]['n_cells']),
        })
    return pd.DataFrame(rec).sort_values('niche_first6').reset_index(drop=True)

### 2-B: Matching Execution Notes

This cell executes alignment for both arms and reports matched-pair counts. These counts are critical for interpreting downstream metrics and variance.

In [ ]:
def get_alignment(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    d = assign_niche_id(df)
    d = d[d['half'].isin(['first6', 'next6'])].copy()

    prob_cols = detect_prob_cols(d)
    expr_cols = detect_expr_cols(d)
    feature_cols = prob_cols + expr_cols

    c1 = niche_centroids(d[d['half'] == 'first6'])
    c2 = niche_centroids(d[d['half'] == 'next6'])

    aln = align_niches(c1, c2, feature_cols=feature_cols)
    return d, aln, feature_cols

baseline_cells, baseline_align, baseline_feat_cols = get_alignment(baseline_df)
fm_cells, fm_align, fm_feat_cols = get_alignment(fm_df)

print('Baseline matched pairs:', len(baseline_align))
print('FM matched pairs:', len(fm_align))
display(baseline_align.head())
display(fm_align.head())

### 2-C: Transition to Quantitative Consistency

After niche pairing is established, we compute marker-, expression-, and composition-level agreement metrics to compare baseline versus FM behavior.

## Stage 3 · Consistency Metrics (No GOEA/LLM Re-derivation)

For each matched niche pair, we compute three complementary agreement signals:

1. Marker overlap (Jaccard)
$$
J(A,B) = \frac{|A \cap B|}{|A \cup B|}
$$
where $A,B$ are top differential marker sets from first6 and next6.

2. Mean-expression correlation
Pearson correlation between niche-level mean expression vectors.

3. Cell-type composition similarity
Cosine similarity between niche-level mean `prob_*` vectors.

Interpretation rule of thumb:
- Higher values indicate stronger cross-half consistency.
- FM improvement is reported as metric delta versus baseline.

In [ ]:
def top_markers_vs_rest(df: pd.DataFrame, niche_id: int, top_n: int = 30) -> list[str]:
    expr_cols = detect_expr_cols(df)
    if not expr_cols:
        return []

    in_grp = df[df['niche_id'] == niche_id]
    out_grp = df[df['niche_id'] != niche_id]

    if len(in_grp) < 3 or len(out_grp) < 3:
        return []

    rec = []
    x1 = in_grp[expr_cols].to_numpy(dtype=float)
    x0 = out_grp[expr_cols].to_numpy(dtype=float)

    m1 = np.nanmean(x1, axis=0)
    m0 = np.nanmean(x0, axis=0)
    logfc = np.log2((m1 + EPS) / (m0 + EPS))

    _, pvals = ttest_ind(x1, x0, axis=0, equal_var=False, nan_policy='omit')
    pvals = np.nan_to_num(pvals, nan=1.0, posinf=1.0, neginf=1.0)

    for gene_col, fc, pv in zip(expr_cols, logfc, pvals):
        rec.append((gene_col, float(fc), float(pv)))

    de = pd.DataFrame(rec, columns=['gene_col', 'log2fc', 'pval'])
    de = de.sort_values(['log2fc', 'pval'], ascending=[False, True])
    return [g.replace('expr_', '') for g in de.head(top_n)['gene_col'].tolist()]

def jaccard(a: list[str], b: list[str]) -> float:
    sa, sb = set(a), set(b)
    if not sa and not sb:
        return np.nan
    return len(sa & sb) / max(1, len(sa | sb))

def comp_similarity(a: pd.Series, b: pd.Series) -> float:
    va = a.to_numpy(dtype=float)
    vb = b.to_numpy(dtype=float)
    na = np.linalg.norm(va) + EPS
    nb = np.linalg.norm(vb) + EPS
    return float(np.dot(va, vb) / (na * nb))

def eval_consistency(cells: pd.DataFrame, alignment: pd.DataFrame, pipeline_name: str) -> pd.DataFrame:
    expr_cols = detect_expr_cols(cells)
    prob_cols = detect_prob_cols(cells)

    d1 = cells[cells['half'] == 'first6']
    d2 = cells[cells['half'] == 'next6']

    out = []
    for _, row in alignment.iterrows():
        n1 = int(row['niche_first6'])
        n2 = int(row['niche_next6'])

        s1 = d1[d1['niche_id'] == n1]
        s2 = d2[d2['niche_id'] == n2]

        m1 = top_markers_vs_rest(d1, niche_id=n1, top_n=TOP_MARKERS)
        m2 = top_markers_vs_rest(d2, niche_id=n2, top_n=TOP_MARKERS)
        marker_j = jaccard(m1, m2)

        if expr_cols and len(s1) > 0 and len(s2) > 0:
            e1 = s1[expr_cols].mean(axis=0)
            e2 = s2[expr_cols].mean(axis=0)
            expr_corr = float(np.corrcoef(e1, e2)[0, 1]) if len(expr_cols) > 1 else np.nan
        else:
            expr_corr = np.nan

        if prob_cols and len(s1) > 0 and len(s2) > 0:
            p1 = s1[prob_cols].mean(axis=0)
            p2 = s2[prob_cols].mean(axis=0)
            comp_cos = comp_similarity(p1, p2)
        else:
            comp_cos = np.nan

        out.append({
            'pipeline': pipeline_name,
            'niche_first6': n1,
            'niche_next6': n2,
            'align_cosine_similarity': float(row['cosine_similarity']),
            'marker_jaccard': marker_j,
            'expr_mean_corr': expr_corr,
            'composition_cosine': comp_cos,
            'n_cells_first6': int(row['n_cells_first6']),
            'n_cells_next6': int(row['n_cells_next6']),
            'top_markers_first6': ','.join(m1[:10]),
            'top_markers_next6': ','.join(m2[:10]),
        })

    return pd.DataFrame(out)

### 3-A: Metric Function Definitions

This block defines reusable functions for marker overlap, expression correlation, and composition similarity. Keeping these modular allows consistent evaluation across both experimental arms.

In [ ]:
baseline_cons = eval_consistency(baseline_cells, baseline_align, 'baseline')
fm_cons = eval_consistency(fm_cells, fm_align, 'foundation_model')

cons_all = pd.concat([baseline_cons, fm_cons], ignore_index=True)
display(cons_all.head())

summary = (cons_all
    .groupby('pipeline', observed=True)[['align_cosine_similarity', 'marker_jaccard', 'expr_mean_corr', 'composition_cosine']]
    .mean(numeric_only=True)
    .reset_index())

print('Pipeline summary (mean consistency metrics):')
display(summary)

### 3-B: Arm-Level Summary Aggregation

This cell runs the metric engine for baseline and FM, concatenates per-pair outputs, and computes arm-level means used in result tables.

## Stage 4 · Reference Labels from reference-crosscohort-label-transfer

GOEA/LLM-derived niche labels are imported as external reference outcomes for reporting convenience only.

This notebook does not:
- regenerate GOEA terms,
- prompt an LLM for renaming,
- or claim methodological novelty in niche naming.

This notebook does:
- attach reference labels to matched niche rows,
- and quantify cross-half consistency under baseline vs FM pipelines.

### 4-A: Reference Label Attachment Logic

This step maps matched niche IDs to human-readable labels from reference-crosscohort-label-transfer, enabling manuscript-friendly reporting without re-running GOEA/LLM.

In [ ]:
def attach_ref_labels(cons_df: pd.DataFrame, ref: pd.DataFrame | None) -> pd.DataFrame:
    out = cons_df.copy()
    if ref is None:
        out['ref_label_first6'] = 'NA'
        return out

    # Expected matching keys can vary. We use 'niche_<id>' as default key.
    key_map = {str(k): v for k, v in zip(ref['niche_key'], ref['ref_label'])}

    def map_label(nid: int) -> str:
        k1 = f'niche_{nid}'
        k2 = str(nid)
        return key_map.get(k1, key_map.get(k2, 'NA'))

    out['ref_label_first6'] = out['niche_first6'].apply(map_label)
    return out

report_df = attach_ref_labels(cons_all, ref_labels)
display(report_df.head())

### 4-B: Optional AUC/CM Evaluation (Classification-Style Diagnostics)

If label-level truth/prediction pairs exist after mapping, we can report confusion matrix and classification metrics.

Notes:
- CM is always computed when label pairs are available.
- AUC requires probabilistic class scores; this cell reports AUC only if suitable score columns exist.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize

# Build optional label-pair table for classification-style diagnostics.
label_diag_df = report_df.copy()

# If reference labels are missing, skip gracefully.
if 'ref_label_first6' not in label_diag_df.columns or label_diag_df['ref_label_first6'].eq('NA').all():
    print('AUC/CM skipped: no usable reference labels in report_df.')
else:
    # Predicted label proxy: use mapped next6 ID through the same label map when available.
    # This is a diagnostic view, not a replacement for full sample-level classifier outputs.
    if ref_labels is not None and {'niche_key', 'ref_label'}.issubset(ref_labels.columns):
        key_map = {str(k): v for k, v in zip(ref_labels['niche_key'], ref_labels['ref_label'])}

        def map_next(nid: int) -> str:
            return key_map.get(f'niche_{nid}', key_map.get(str(nid), 'NA'))

        label_diag_df['pred_label_proxy'] = label_diag_df['niche_next6'].apply(map_next)
        eval_df = label_diag_df[(label_diag_df['ref_label_first6'] != 'NA') & (label_diag_df['pred_label_proxy'] != 'NA')].copy()
    else:
        eval_df = pd.DataFrame()

    if eval_df.empty:
        print('AUC/CM skipped: no valid label pairs after proxy mapping.')
    else:
        y_true = eval_df['ref_label_first6'].astype(str).to_numpy()
        y_pred = eval_df['pred_label_proxy'].astype(str).to_numpy()
        labels = sorted(pd.unique(np.concatenate([y_true, y_pred])))

        cm = confusion_matrix(y_true, y_pred, labels=labels)
        print('Confusion matrix labels:', labels)
        print(cm)
        print('Accuracy:', round(float(accuracy_score(y_true, y_pred)), 4))
        print('Macro-F1:', round(float(f1_score(y_true, y_pred, average='macro', zero_division=0)), 4))
        print('\nClassification report:')
        print(classification_report(y_true, y_pred, zero_division=0))

        # Optional AUC only if score columns exist (prob_* style). Usually absent in this notebook.
        score_cols = [c for c in eval_df.columns if c.startswith('prob_')]
        if score_cols:
            y_bin = label_binarize(y_true, classes=labels)
            y_score = eval_df[score_cols].to_numpy(dtype=float)
            if y_score.shape[1] == len(labels):
                try:
                    auc_macro = roc_auc_score(y_bin, y_score, average='macro', multi_class='ovr')
                    print('Macro AUC (OVR):', round(float(auc_macro), 4))
                except Exception as e:
                    print('AUC skipped due to scoring-shape issue:', e)
            else:
                print('AUC skipped: score column count does not match class count.')
        else:
            print('AUC skipped: no probabilistic score columns found (CM/F1 still reported).')

### 4-C: Export Artifacts for Manuscript Tables

This cell writes machine-readable summary tables that can be imported directly into plotting scripts, supplementary materials, or reporting pipelines.

In [ ]:
# Export tables for manuscript figures/supplement.
OUTDIR = PROJECT_ROOT / 'consistency-report'
OUTDIR.mkdir(parents=True, exist_ok=True)

summary.to_csv(OUTDIR / 'pipeline_summary.csv', index=False)
report_df.to_csv(OUTDIR / 'niche_pair_consistency.csv', index=False)

print('Wrote:')
print('-', OUTDIR / 'pipeline_summary.csv')
print('-', OUTDIR / 'niche_pair_consistency.csv')

### 4-D: Visualization Layer

This figure compares baseline vs FM distributions for each consistency metric. Use it as the high-level result panel for the cross-half robustness claim.

In [ ]:
# Visual summary
plot_cols = ['align_cosine_similarity', 'marker_jaccard', 'expr_mean_corr', 'composition_cosine']
fig, axes = plt.subplots(1, len(plot_cols), figsize=(5 * len(plot_cols), 5), constrained_layout=True)

for ax, col in zip(axes, plot_cols):
    sns.boxplot(data=report_df, x='pipeline', y=col, ax=ax)
    sns.stripplot(data=report_df, x='pipeline', y=col, ax=ax, color='black', alpha=0.5)
    ax.set_title(col)
    ax.set_xlabel('')

plt.suptitle('Cross-half niche consistency: baseline vs foundation model')
plt.show()

## Paper Methods Text (Suggested, Non-duplicative Scope)

### Methods (for manuscript)
Niche annotations used in this study were obtained from the previously established reference-crosscohort-label-transfer annotation outcomes. We did not re-derive or modify the GOEA+LLM labeling procedure in the present work. Instead, those outcomes were treated as fixed reference labels.

The current analysis was designed to quantify cross-half consistency and the impact of foundation-model-assisted niche discovery. The cohort contained 12 tissue portions (4 samples x Top/Mid/Bot), split into first6 (P1-P6) and next6 (P7-P12). For each split, we evaluated two pipelines under matched settings: (i) composition-only niche features (baseline), and (ii) composition plus foundation-model morphology features (Hoptimus).

Because niche IDs are not guaranteed to align across halves, we performed niche pairing using Hungarian matching on centroid similarity. For each matched pair, we computed marker overlap (Jaccard of top differential markers), mean-expression correlation, and cell-type composition cosine similarity. Improvement attributable to foundation-model features was summarized as the metric delta between the two pipelines.

### Scope statement (optional sentence)
This work evaluates label consistency and transfer robustness; methodological innovation for GOEA+LLM niche naming is attributed to reference-crosscohort-label-transfer and is not duplicated here.

## Stage 5 · How to Read the Outputs

### 5-A: Core Tables
- `pipeline_summary.csv`: mean metrics per pipeline.
- `niche_pair_consistency.csv`: per-matched-pair metrics with optional reference labels.

### 5-B: Core Figure
The box/strip plot compares baseline vs FM on each metric. If FM boxes are consistently higher, this supports improved transfer consistency.

### 5-C: Paper Positioning
Use these outcomes as evidence of robustness and transfer consistency. Attribute GOEA+LLM methodological innovation to reference-crosscohort-label-transfer, and cite it rather than re-describing the full pipeline here.

## Stage 6 · Limitations and Reviewer FAQ

### 6-A: Study Limitations

1. Split size is modest.
The first6/next6 design (6 portions each) supports paired consistency analysis, but confidence intervals can still be wide for rare niches.

2. Niche alignment is similarity-based, not identity-based.
Hungarian matching improves comparability but does not prove biological identity in a causal sense.

3. Expression availability may vary.
If `expr_*` channels are sparse or partially missing, marker-based consistency becomes less stable and should be interpreted with caution.

4. FM gains may be tissue-context dependent.
Foundation-model benefit can differ by morphology complexity, cellular density, and staining variation.

5. This notebook is validation-focused.
GOEA/LLM naming methodology is intentionally not re-derived here; novelty attribution remains with reference-crosscohort-label-transfer.

---

### 6-B: Reviewer FAQ (Suggested Responses)

**Q1. Why not compare niche IDs directly across halves?**
A1. IDs are generated independently and can be permuted. We therefore use one-to-one centroid matching before consistency scoring.

**Q2. Is there label leakage from GOEA/LLM?**
A2. No. GOEA/LLM outcomes are imported as fixed references only. This notebook does not re-train or re-tune naming models.

**Q3. How do you ensure FM vs baseline fairness?**
A3. Both arms share the same cohort split and matched niche settings; only FM feature inclusion differs.

**Q4. Could FM improvements be driven by larger feature dimension alone?**
A4. We report paired metric deltas per matched niche and recommend an additional sensitivity run with controlled dimensionality (e.g., PCA-matched baseline) in supplementary analysis.

**Q5. What is the primary claim supported by this notebook?**
A5. Foundation-model-assisted niche discovery improves cross-half consistency under the defined quantitative metrics.

---

### 6-C: Optional Supplementary Checks

- Bootstrap confidence intervals for each consistency metric.
- Repeat runs with multiple seeds and report mean ± SD.
- Leave-one-sample-out split test (e.g., S1/S3 train-side vs S2/S4 validation-side variants).
- Stratified analysis by niche prevalence (common vs rare niches).